In [1]:
from unike.utils import Link
from unike.module.model import RotatE, ComplEx
import pandas as pd

In [2]:
ent_tol = 151321
rel_tol = 31

In [3]:
rotate_model = RotatE(
    ent_tol=ent_tol,
    rel_tol=rel_tol,
    dim=256,
    margin=6.0,
    epsilon=2.0,
)

In [ ]:
rotate_model.load_checkpoint('checkpoints/rotate/all/rotate-100.pth')

In [4]:
complex_model = ComplEx(
    ent_tol=ent_tol,
    rel_tol=rel_tol,
    dim=200
)

In [ ]:
complex_model.load_checkpoint('checkpoints/complex/all/complex-100.pth')

In [5]:
link = Link(
    in_path='data',
    model=rotate_model
)

In [6]:
dmd_ent_id = [27017]
drug_dis_rel_id = [3, 4]
all_drug_ent_id = [link.ent2id[ent_name] for ent_name in link.ent2id.keys() if ent_name.split(':')[-1] == 'drug']
len(all_drug_ent_id)

9542

In [7]:
nodes = pd.read_csv('../../../kg/nodes.csv')
nodes

,node_index,node_id,node_type,node_name,node_source
0,0,kg4rd:381,gene/protein,ARF5,NCBI
1,1,kg4rd:4074,gene/protein,M6PR,NCBI
2,2,kg4rd:2288,gene/protein,FKBP4,NCBI
3,3,kg4rd:56603,gene/protein,CYP26B1,NCBI
4,4,kg4rd:55471,gene/protein,NDUFAF7,NCBI
...,...,...,...,...,...
151316,151316,kg4rd:R-HSA-9694322,pathway,Virion Assembly and Release,REACTOME
151317,151317,kg4rd:R-HSA-9727281,pathway,Translation of Accessory Proteins,REACTOME
151318,151318,kg4rd:R-HSA-9828721,pathway,Translation of respiratory syncytial virus mRNAs,REACTOME
151319,151319,kg4rd:1062,anatomy,anatomical entity,UBERON


In [ ]:
df = link.link(dmd_ent_id, drug_dis_rel_id, all_drug_ent_id, device='cuda:3').drop(columns=['score'])
df = pd.merge(df, nodes, left_on=['tail'], right_on=['node_index'], how='left').drop(columns=['node_index', 'node_type', 'node_name']).rename(columns={'node_id': 'tail_id', 'node_source': 'tail_source'})
df.to_csv('link_result_rotate_100.csv', index=False)

In [ ]:
link.model = complex_model
df = link.link(dmd_ent_id, drug_dis_rel_id, all_drug_ent_id, device='cuda:3').drop(columns=['score'])
df = pd.merge(df, nodes, left_on=['tail'], right_on=['node_index'], how='left').drop(columns=['node_index', 'node_type', 'node_name']).rename(columns={'node_id': 'tail_id', 'node_source': 'tail_source'})
df.to_csv('link_result_complex_100.csv', index=False)